In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph

In [ ]:
!pip install -qU "langchain[groq]"

In [ ]:
!pip install --upgrade --quiet langgraph langchain-community beautifulsoup4

In [ ]:
!pip install -qU langchain-huggingface

In [ ]:
!pip install langchain langchain-community

In [ ]:
!pip install pypdf

In [ ]:
!pip install --upgrade --quiet langgraph langchain-community beautifulsoup4

In [ ]:
!pip install groq

In [ ]:
!pip install --upgrade langchain langchain-core langchain-community

In [ ]:
!pip install -qU "langchain[groq]"

In [ ]:
!pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph

In [ ]:
!pip install langchain langchain-community

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from operator import add as add_messages
from langchain_core.messages import HumanMessage, AIMessage
import json
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import ToolMessage # Passes data back to LLM after it calls a tool such as the content and the tool_call_id
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

In [ ]:
import getpass
import os

if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("llama3-8b-8192", model_provider="groq")

Enter API key for Groq: ··········


In [ ]:
# job_matcher.py
from sentence_transformers import SentenceTransformer, util
import json

# Load model (downloads once, then runs offline)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Load job data
with open('jobs.json', 'r', encoding='utf-8') as f:
    jobs = json.load(f)

job_texts = [
    f"{j.get('title','')} at {j.get('company','')} in {j.get('location','')}. "
    f"Skills: {', '.join(j.get('skills',[]))}. {j.get('description','')}"
    for j in jobs
]

# encode jobs once
job_embeddings = model.encode(job_texts, convert_to_tensor=True)

def match_jobs(user_text, top_k=5):
    query_emb = model.encode(user_text, convert_to_tensor=True)
    scores = util.cos_sim(query_emb, job_embeddings)[0]
    top_k_idx = scores.argsort(descending=True)[:top_k]
    results = []
    for idx in top_k_idx:
        j = jobs[int(idx)]
        score = float(scores[idx])  # cosine similarity
        results.append({
            "title": j.get("title"),
            "company": j.get("company"),
            "location": j.get("location"),
            "score": round(score * 100, 2),
            "description": j.get("description")
        })
    return results


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing_extensions import List, TypedDict



# Initialize document_loader in the global scope
def load_cv(Data_path):
  def load_documents():
      loader = PyPDFLoader(DATA_PATH)
      return loader.load()
  cv= load_documents()[0].page_content
  return cv

cv = load_cv()
print(cv)

. 
FIRST LAST 
San Francisco, California 94109 | (480) 123‐5689 | sampleresume@gmail.com | linkedin.com/in/sampleresume 
 
SUMMARY  
 
An analytical and results‐driven software engineer with experience in application development, scripting and coding, 
automation, web application design, product testing and deployment, UI testing, and requirements gathering.  Proven 
aptitude for implementing innovative solutions to streamline and automate processes, enhance efficiency, improve 
customer satisfaction, and achieve financial savings. 
 
EDUCATION  
 
UNIVERSITY OF ARIZONA, Tucson, Arizona 
M.S., Computer Science, 2012 
B.S.B.A., Management Information Systems, 2011 
 
TECHNICAL   SKILLS  
 
JavaScript:   ReactJS, AngularJS 1.x, ExpressJS, NodeJS, jQuery, HTML/CSS 
Mobile:   React Native, ExponentJS 
Java:   Spring, Maven 
Databases:   MongoDB, SQL 
Build/Deploy:   Docker, Tomcat, Grunt, Heroku, CircleCI 
 
EXPERIENCE  
 
WALMART, INC., Bentonville, Arkansas 
Programmer Analyst, Call Cent

In [ ]:
from langchain_groq import ChatGroq

def key_adia():
  base_instruction = ("You are an AI that reads a CV and extracts only the key ideas representing the candidate’s main skills, specialties, and expertise areas, returning them as a comma-separated list without numbers or extra text."
      f"cv:{cv}")

  llm = ChatGroq(model="llama-3.1-8b-instant")
  response = llm.invoke(base_instruction)
  return response.content
print(key_adia())

JavaScript, ReactJS, AngularJS, NodeJS, ExpressJS, HTML/CSS, React Native, ExponentJS, Java, Spring, Maven, MongoDB, SQL, Docker, Tomcat, Grunt, Heroku, CircleCI, Socket.io, YouTube API, MySQL, PassportJS, Google Maps, Accuweather


In [ ]:

results = match_jobs(response.content, top_k=3)
print(results)


[{'title': 'Full Stack Developer', 'company': 'KairoSoft', 'location': 'Cairo', 'score': 56.56, 'description': 'Develop full stack web apps integrating front and back ends.'}, {'title': 'Frontend Developer', 'company': 'TuniTech', 'location': 'Tunis', 'score': 54.17, 'description': 'Build responsive web applications using React and modern JS frameworks.'}, {'title': 'Backend Developer', 'company': 'Casabyte', 'location': 'Casablanca', 'score': 49.21, 'description': 'Design and maintain backend services and APIs for scalable systems.'}]


In [ ]:

def job_finde_cv(path):
   cv=load_cv(path)
   key_cv=key_adia()
   results = match_jobs(key_cv, top_k=3)
   return results
print(job_finde_cv())


[{'title': 'Frontend Developer', 'company': 'TuniTech', 'location': 'Tunis', 'score': 60.43, 'description': 'Build responsive web applications using React and modern JS frameworks.'}, {'title': 'Full Stack Developer', 'company': 'KairoSoft', 'location': 'Cairo', 'score': 55.38, 'description': 'Develop full stack web apps integrating front and back ends.'}, {'title': 'Backend Intern', 'company': 'CodeAfrica', 'location': 'Tunis', 'score': 54.21, 'description': 'Assist backend team in developing APIs and database queries.'}]


In [ ]:
#API

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

app = FastAPI()



# ----------------------------
# API Models
# ----------------------------
class State(BaseModel):
    data: dict


# ----------------------------
# API Endpoint
# ----------------------------
@app.post("/job_serch")
def job_serch( State):
  state=job_finde_cv(state["path"])
  return state


# ----------------------------
# Run server
# ----------------------------
if __name__ == "__main__":
    uvicorn.run("api:app", host="0.0.0.0", port=8000, reload=True)